# 03 — Agrupación · Google Cloud Compute Engine (L4)

**Este notebook es para ejecutar en el VM de Google Cloud — no en Colab.**

**Tercer notebook de la trilogía Vena:**
- NB01 Clasificación → detecta si el paciente está sano o enfermo
- NB02 Regresión    → estima la edad del paciente desde la radiografía
- **NB03 Agrupación → descubre patrones ocultos combinando visión + edad + género**

**Estrategia:**
1. Cargar el backbone de NB01 (DenseNet121 ya entrenado en 96k radiografías)
2. Extraer embeddings de 1024 dimensiones por imagen — la 'huella visual' de cada RX
3. Combinar con metadata clínica: edad y género
4. Reducir dimensionalidad con PCA
5. **K-Means** — clusters con K óptimo por método del codo + Silhouette
6. **DBSCAN** — clusters sin definir K, el algoritmo decide
7. Visualizar ambos con UMAP
8. Mostrar radiografías reales de cada cluster
9. Comparar K-Means vs DBSCAN

---
## INSTRUCCIONES — leer antes de ejecutar

### Prerequisito
El NB01 debe haber terminado y guardado el backbone en:
`/home/user/checkpoints/backbone_densenet121_v4.keras`

### Ejecutar
```bash
nohup jupyter nbconvert --to notebook --execute --inplace 03_Agrupacion_Vena.ipynb > agrupacion.log 2>&1 &
```

### Al terminar, subir resultados al bucket
```bash
gsutil -m cp -r /home/user/checkpoints/agrupacion/ gs://vena-dataset/resultados_agrupacion/
```

## Celda 1 — Rutas y verificación de archivos

In [ ]:
from pathlib import Path

RUTA_BASE       = Path('/home/user/data/DatasetV2')
RUTA_BACKBONE   = Path('/home/user/checkpoints/backbone_densenet121_v4.keras')
RUTA_CSV        = RUTA_BASE / 'master_train.csv'
RUTA_RESULTADOS = Path('/home/user/checkpoints/agrupacion')
RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)

assert RUTA_BASE.exists(),     f'No se encontro DatasetV2 en {RUTA_BASE}'
assert RUTA_BACKBONE.exists(), f'No se encontro backbone en {RUTA_BACKBONE}\nEjecutar NB01 primero.'
assert RUTA_CSV.exists(),      f'No se encontro master_train.csv en {RUTA_CSV}'

print('Rutas verificadas correctamente.')
print(f'  Dataset    : {RUTA_BASE}')
print(f'  Backbone   : {RUTA_BACKBONE}')
print(f'  Resultados : {RUTA_RESULTADOS}')

## Celda 2 — Verificación de GPU

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'GPU encontrada: {gpus[0].name}')
    print('El notebook usara aceleracion GPU.')
else:
    print('ERROR: No se encontro GPU.')
    print('Verificar que el VM fue creado con acelerador L4.')

## Celda 3 — Instalación de dependencias e importaciones

UMAP no viene preinstalado — se instala aqui antes de importar.

In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'umap-learn', '--quiet'], check=True)
print('umap-learn instalado.')

In [ ]:
import numpy              as np
import pandas             as pd
import matplotlib.pyplot  as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn            as sns
import time
import warnings
warnings.filterwarnings('ignore')

from pathlib                               import Path
from PIL                                   import Image
from tensorflow.keras.models               import load_model, Model
from tensorflow.keras.layers               import GlobalAveragePooling2D
from tensorflow.keras.preprocessing.image  import ImageDataGenerator
from sklearn.preprocessing                 import StandardScaler
from sklearn.decomposition                 import PCA
from sklearn.cluster                       import MiniBatchKMeans, DBSCAN
from sklearn.metrics                       import silhouette_score, davies_bouldin_score
from sklearn.neighbors                     import NearestNeighbors
import umap

# ── Hiperparámetros ────────────────────────────────────────────────────────────
TAMANO_IMAGEN  = 224
BATCH_SIZE     = 64
SEMILLA        = 42
N_PCA          = 64       # Dimensiones PCA antes de clustering
N_CLUSTERS_MIN = 3        # Rango K-Means a evaluar
N_CLUSTERS_MAX = 10
PESO_META      = 10.0     # Peso de edad/genero vs 1024 dims visuales
N_MUESTRAS_RX  = 6        # Radiografias a mostrar por cluster

np.random.seed(SEMILLA)
tf.random.set_seed(SEMILLA)
print('Librerias cargadas correctamente.')
print(f'TensorFlow: {tf.__version__}')

## Celda 4 — Cargar master_train.csv

Solo se usa el conjunto de entrenamiento (67k imagenes).
Val y test no se tocan — deben permanecer virgenes para evaluacion del modelo.

In [ ]:
df = pd.read_csv(RUTA_CSV)

# Construir ruta absoluta en la VM
df['ruta_completa'] = df['path'].apply(lambda p: str(RUTA_BASE / p))

# Normalizar genero
df['patient_gender'] = df['patient_gender'].str.strip().str.upper().map(
    {'M': 'M', 'MALE': 'M', 'F': 'F', 'FEMALE': 'F'}
)

# Eliminar filas con metadata nula
antes = len(df)
df = df.dropna(subset=['patient_gender', 'patient_age', 'label']).reset_index(drop=True)

print(f'Registros cargados  : {len(df):,}  (eliminados por nulos: {antes - len(df)})')
print(f'Sanos       (0)     : {(df["label"]==0).sum():,} ({(df["label"]==0).mean()*100:.1f}%)')
print(f'Enfermos    (1)     : {(df["label"]==1).sum():,} ({(df["label"]==1).mean()*100:.1f}%)')
print(f'Hombres     (M)     : {(df["patient_gender"]=="M").sum():,} ({(df["patient_gender"]=="M").mean()*100:.1f}%)')
print(f'Mujeres     (F)     : {(df["patient_gender"]=="F").sum():,} ({(df["patient_gender"]=="F").mean()*100:.1f}%)')
print(f'Edad media          : {df["patient_age"].mean():.1f} años  (min: {df["patient_age"].min()}, max: {df["patient_age"].max()})')
print(f'Fuente NIH          : {(df["source"]=="NIH").sum():,}')
print(f'Fuente CheXpert     : {(df["source"]=="CHEX").sum():,}')
df.head(3)

## Celda 5 — Cargar backbone y construir extractor de embeddings

El backbone exportado por NB01 termina antes del GlobalAveragePooling2D
(salida: `(None, 7, 7, 1024)`). Se agrega aqui para obtener un vector
de 1024 dimensiones por imagen — la 'huella visual' de cada radiografia.

In [ ]:
print('Cargando backbone...')
backbone_raw = load_model(str(RUTA_BACKBONE), compile=False)

print(f'Backbone cargado.')
print(f'  Entrada : {backbone_raw.input_shape}')
print(f'  Salida  : {backbone_raw.output_shape}')

# Agregar GlobalAveragePooling2D para obtener vector 1D de 1024 features
embedding_output = GlobalAveragePooling2D(name='embedding')(backbone_raw.output)
extractor = Model(
    inputs  = backbone_raw.input,
    outputs = embedding_output,
    name    = 'extractor_embeddings'
)
extractor.trainable = False

print(f'\nExtractor de embeddings listo.')
print(f'  Salida embedding: {extractor.output_shape}')
print(f'  <- 1024 features por radiografia')

## Celda 6 — Extraer embeddings

Se procesan las 67k imagenes en batches de 64.
Si los embeddings ya existen en disco (sesion reiniciada), se cargan directamente.

In [ ]:
ruta_emb = RUTA_RESULTADOS / 'embeddings_train.npy'

if ruta_emb.exists():
    embeddings = np.load(str(ruta_emb))
    print(f'Embeddings cargados desde disco: {embeddings.shape}')
else:
    gen = ImageDataGenerator(rescale=1./255)
    flujo = gen.flow_from_dataframe(
        dataframe   = df,
        x_col       = 'ruta_completa',
        y_col       = 'label',
        target_size = (TAMANO_IMAGEN, TAMANO_IMAGEN),
        batch_size  = BATCH_SIZE,
        class_mode  = 'raw',
        shuffle     = False,  # CRITICO: mantener orden con df
        color_mode  = 'rgb'
    )

    print(f'Extrayendo embeddings de {len(df):,} radiografias...')
    print(f'Batches: {len(flujo)} x {BATCH_SIZE} imagenes')
    t0 = time.time()

    embeddings = extractor.predict(flujo, verbose=1)
    embeddings = embeddings[:len(df)]  # recortar padding del generador

    print(f'\nExtraccion completada en {time.time()-t0:.1f}s')
    print(f'Shape embeddings: {embeddings.shape}')

    np.save(str(ruta_emb), embeddings)
    print(f'Embeddings guardados en: {ruta_emb}')

## Celda 7 — Construir matriz de features combinada

Se combinan los embeddings visuales con la metadata clinica.
La edad y el genero se amplifican con PESO_META para que no queden
opacados por las 1024 dimensiones visuales.

In [ ]:
# Normalizar edad a [0, 1]
edad_min  = df['patient_age'].min()
edad_max  = df['patient_age'].max()
edad_norm = (df['patient_age'].values - edad_min) / (edad_max - edad_min)

# Codificar genero: M=0, F=1
genero_cod = (df['patient_gender'] == 'F').astype(float).values

# Escalar embeddings
scaler     = StandardScaler()
emb_scaled = scaler.fit_transform(embeddings)

# Concatenar: embeddings + edad + genero
X = np.hstack([
    emb_scaled,
    edad_norm.reshape(-1, 1)  * PESO_META,
    genero_cod.reshape(-1, 1) * PESO_META
])

print(f'Matriz de features construida: {X.shape}')
print(f'  {embeddings.shape[1]} features visuales')
print(f'  + edad (peso x{PESO_META})')
print(f'  + genero (peso x{PESO_META})')
print(f'  = {X.shape[1]} dimensiones totales')

:## Celda 8 — Reduccion de dimensionalidad con PCA

PCA reduce de 1026 a 64 dimensiones antes del clustering.
Esto elimina ruido, acelera K-Means y DBSCAN, y retiene
la mayor parte de la varianza informativa.

In [ ]:
print(f'Reduciendo de {X.shape[1]} a {N_PCA} dimensiones con PCA...')
t0    = time.time()
pca   = PCA(n_components=N_PCA, random_state=SEMILLA)
X_pca = pca.fit_transform(X)

varianza = pca.explained_variance_ratio_.cumsum()[-1]
print(f'Completado en {time.time()-t0:.1f}s')
print(f'Varianza explicada con {N_PCA} componentes: {varianza*100:.1f}%')
print(f'Shape X_pca: {X_pca.shape}')

# Curva de varianza acumulada
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(np.cumsum(pca.explained_variance_ratio_) * 100, color='steelblue', linewidth=2)
ax.axvline(x=N_PCA, color='red', linestyle='--', alpha=0.7, label=f'{N_PCA} componentes ({varianza*100:.1f}%)')
ax.set_xlabel('Numero de componentes PCA')
ax.set_ylabel('Varianza explicada acumulada (%)')
ax.set_title('Varianza explicada por PCA')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(RUTA_RESULTADOS / 'pca_varianza.png'), dpi=150, bbox_inches='tight')
plt.show()

## Celda 9 — K-Means: determinar K optimo

Se evaluan dos criterios para cada K:
- **Inercia** (metodo del codo): suma de distancias al centroide. Baja a medida que K sube.
  El 'codo' es el punto donde la ganancia marginal cae — el K optimo.
- **Silhouette score**: mide que tan bien separados estan los clusters.
  Rango [-1, 1]. Mas alto es mejor. Se calcula sobre muestra de 5000 puntos.

In [ ]:
inercias    = []
silhouettes = []
ks          = list(range(N_CLUSTERS_MIN, N_CLUSTERS_MAX + 1))

print(f'Evaluando K de {N_CLUSTERS_MIN} a {N_CLUSTERS_MAX}...')
print(f'{"K":>3}  {"Inercia":>12}  {"Silhouette":>10}  {"Tiempo":>8}')
print('-' * 40)

for k in ks:
    t0  = time.time()
    km  = MiniBatchKMeans(n_clusters=k, random_state=SEMILLA, batch_size=4096, n_init=10)
    lbl = km.fit_predict(X_pca)
    inercias.append(km.inertia_)

    idx = np.random.choice(len(X_pca), size=min(5000, len(X_pca)), replace=False)
    sil = silhouette_score(X_pca[idx], lbl[idx], random_state=SEMILLA)
    silhouettes.append(sil)
    print(f'{k:>3}  {km.inertia_:>12.0f}  {sil:>10.4f}  {time.time()-t0:>7.1f}s')

K_OPTIMO_SUGERIDO = ks[np.argmax(silhouettes)]
print(f'\nK con mejor Silhouette: {K_OPTIMO_SUGERIDO}')

# Grafica
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(ks, inercias, 'o-', color='steelblue', linewidth=2, markersize=8)
ax1.set_xlabel('Numero de clusters (K)', fontsize=12)
ax1.set_ylabel('Inercia', fontsize=12)
ax1.set_title('Metodo del Codo', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3)

ax2.plot(ks, silhouettes, 's-', color='coral', linewidth=2, markersize=8)
ax2.axvline(x=K_OPTIMO_SUGERIDO, color='green', linestyle='--', alpha=0.7,
            label=f'K optimo = {K_OPTIMO_SUGERIDO}')
ax2.set_xlabel('Numero de clusters (K)', fontsize=12)
ax2.set_ylabel('Silhouette Score', fontsize=12)
ax2.set_title('Silhouette Score por K', fontsize=13, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.suptitle('Seleccion del numero optimo de clusters — K-Means', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(RUTA_RESULTADOS / 'kmeans_seleccion_k.png'), dpi=150, bbox_inches='tight')
plt.show()

## Celda 10 — K-Means: entrenar con K optimo

*   List item
*   List item



Ajusta `K_OPTIMO` manualmente si la grafica anterior sugiere un K diferente
al seleccionado automaticamente por Silhouette.

In [ ]:
K_OPTIMO = K_OPTIMO_SUGERIDO  # <- ajustar manualmente si es necesario

print(f'Entrenando K-Means final con K={K_OPTIMO}...')
t0 = time.time()
kmeans = MiniBatchKMeans(
    n_clusters   = K_OPTIMO,
    random_state = SEMILLA,
    batch_size   = 4096,
    n_init       = 20
)
df['cluster_kmeans'] = kmeans.fit_predict(X_pca)
print(f'Completado en {time.time()-t0:.1f}s')

print(f'\nDistribucion de clusters K-Means:')
for k in range(K_OPTIMO):
    n = (df['cluster_kmeans'] == k).sum()
    print(f'  Cluster {k}: {n:,} ({n/len(df)*100:.1f}%)')

## Celda 11 — DBSCAN: determinar epsilon optimo




DBSCAN necesita dos parametros:
- `eps`: radio de vecindad. Se determina con el grafico k-NN distance.
- `min_samples`: minimo de puntos para formar un cluster.

El 'codo' en la curva k-NN distance indica el epsilon optimo.
Puntos por encima del codo son ruido (outliers).

In [ ]:
print('Calculando distancias k-NN para seleccionar epsilon de DBSCAN...')
print('(Usando muestra de 10,000 puntos para agilizar)')

idx_muestra = np.random.choice(len(X_pca), size=min(10000, len(X_pca)), replace=False)
X_muestra   = X_pca[idx_muestra]

MIN_SAMPLES = max(5, int(np.log(len(X_pca))))  # heuristica: log(n)
print(f'min_samples = {MIN_SAMPLES} (heuristica log(n))')

t0  = time.time()
nbrs = NearestNeighbors(n_neighbors=MIN_SAMPLES).fit(X_muestra)
distancias, _ = nbrs.kneighbors(X_muestra)
distancias_k  = np.sort(distancias[:, -1])[::-1]
print(f'Completado en {time.time()-t0:.1f}s')

# Grafica k-NN distance
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(distancias_k, color='steelblue', linewidth=1.5)
ax.set_xlabel('Puntos ordenados por distancia', fontsize=12)
ax.set_ylabel(f'Distancia al {MIN_SAMPLES}-esimo vecino', fontsize=12)
ax.set_title('Grafico k-NN Distance — Seleccion de epsilon para DBSCAN',
             fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(RUTA_RESULTADOS / 'dbscan_epsilon.png'), dpi=150, bbox_inches='tight')
plt.show()

# Sugerir epsilon en el percentil 90
EPS_SUGERIDO = float(np.percentile(distancias_k, 10))
print(f'\nEpsilon sugerido (percentil 10): {EPS_SUGERIDO:.4f}')
print('Ajustar EPS en la celda siguiente si el codo visual sugiere otro valor.')

## Celda 12 — DBSCAN: entrenar




DBSCAN asigna -1 a los puntos que no pertenecen a ningun cluster (ruido/outliers).
Un porcentaje alto de ruido indica que eps es muy pequeno — aumentarlo.
Un solo cluster gigante indica que eps es muy grande — reducirlo.

In [ ]:
EPS         = EPS_SUGERIDO  # <- ajustar manualmente si es necesario
MIN_SAMPLES = max(5, int(np.log(len(X_pca))))

print(f'Entrenando DBSCAN con eps={EPS:.4f}, min_samples={MIN_SAMPLES}...')
print('(Puede tardar varios minutos con 67k puntos)')
t0 = time.time()

dbscan = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES, n_jobs=-1)
df['cluster_dbscan'] = dbscan.fit_predict(X_pca)

n_clusters_db = len(set(df['cluster_dbscan'])) - (1 if -1 in df['cluster_dbscan'].values else 0)
n_ruido       = (df['cluster_dbscan'] == -1).sum()

print(f'Completado en {time.time()-t0:.1f}s')
print(f'\nResultados DBSCAN:')
print(f'  Clusters encontrados : {n_clusters_db}')
print(f'  Puntos ruido (-1)    : {n_ruido:,} ({n_ruido/len(df)*100:.1f}%)')
print(f'\nDistribucion:')
for k in sorted(df['cluster_dbscan'].unique()):
    n = (df['cluster_dbscan'] == k).sum()
    label = 'RUIDO' if k == -1 else f'Cluster {k}'
    print(f'  {label}: {n:,} ({n/len(df)*100:.1f}%)')

## Celda 13 — UMAP: reduccion a 2D para visualizacion

UMAP proyecta las 64 dimensiones PCA a 2D preservando la estructura local.
El resultado permite visualizar clusters que no son visibles en alta dimension.
Puede tardar ~5-10 minutos con 67k puntos.

In [ ]:
ruta_umap = RUTA_RESULTADOS / 'umap_2d.npy'

if ruta_umap.exists():
    X_umap = np.load(str(ruta_umap))
    print(f'UMAP cargado desde disco: {X_umap.shape}')
else:
    print('Calculando UMAP 2D... (puede tardar 5-10 min)')
    t0 = time.time()
    reducer = umap.UMAP(
        n_components = 2,
        random_state = SEMILLA,
        n_neighbors  = 30,
        min_dist     = 0.1,
        n_jobs       = -1
    )
    X_umap = reducer.fit_transform(X_pca)
    print(f'UMAP completado en {time.time()-t0:.1f}s')
    np.save(str(ruta_umap), X_umap)
    print(f'UMAP guardado en: {ruta_umap}')

df['umap_x'] = X_umap[:, 0]
df['umap_y'] = X_umap[:, 1]

## Celda 14 — Visualizacion UMAP: K-Means vs DBSCAN

Cuatro graficas en un solo panel:
1. Clusters K-Means
2. Clusters DBSCAN
3. Sano vs Enfermo
4. Genero

In [ ]:
fig = plt.figure(figsize=(20, 16))
gs  = gridspec.GridSpec(2, 2, hspace=0.35, wspace=0.3)
axes = [fig.add_subplot(gs[i, j]) for i in range(2) for j in range(2)]

paleta_km = sns.color_palette('tab10', K_OPTIMO)
n_db      = len(set(df['cluster_dbscan'].unique()) - {-1})
paleta_db = sns.color_palette('Set2', max(n_db, 1))

# ── Plot 1: K-Means ──────────────────────────────────────────────────────────
for k in range(K_OPTIMO):
    mask = df['cluster_kmeans'] == k
    axes[0].scatter(df.loc[mask, 'umap_x'], df.loc[mask, 'umap_y'],
                    c=[paleta_km[k]], s=1.5, alpha=0.4, label=f'Cluster {k}')
axes[0].set_title(f'K-Means  (K={K_OPTIMO})', fontsize=13, fontweight='bold')
axes[0].legend(markerscale=6, fontsize=9, loc='upper right')

# ── Plot 2: DBSCAN ───────────────────────────────────────────────────────────
clusters_db = sorted([c for c in df['cluster_dbscan'].unique() if c != -1])
# Ruido primero en gris
mask_ruido = df['cluster_dbscan'] == -1
if mask_ruido.any():
    axes[1].scatter(df.loc[mask_ruido, 'umap_x'], df.loc[mask_ruido, 'umap_y'],
                    c='lightgray', s=1, alpha=0.3, label=f'Ruido ({mask_ruido.sum():,})')
for i, k in enumerate(clusters_db):
    mask = df['cluster_dbscan'] == k
    axes[1].scatter(df.loc[mask, 'umap_x'], df.loc[mask, 'umap_y'],
                    c=[paleta_db[i % len(paleta_db)]], s=1.5, alpha=0.5,
                    label=f'Cluster {k}')
axes[1].set_title(f'DBSCAN  ({n_db} clusters)', fontsize=13, fontweight='bold')
axes[1].legend(markerscale=6, fontsize=9, loc='upper right')

# ── Plot 3: Sano vs Enfermo ──────────────────────────────────────────────────
for lbl, color, nombre in [(0, '#2ecc71', 'Sano'), (1, '#e74c3c', 'Enfermo')]:
    mask = df['label'] == lbl
    axes[2].scatter(df.loc[mask, 'umap_x'], df.loc[mask, 'umap_y'],
                    c=color, s=1.5, alpha=0.35, label=f'{nombre} ({mask.sum():,})')
axes[2].set_title('Sano vs Enfermo', fontsize=13, fontweight='bold')
axes[2].legend(markerscale=6, fontsize=10)

# ── Plot 4: Genero ───────────────────────────────────────────────────────────
for gen, color, nombre in [('M', '#3498db', 'Hombre'), ('F', '#e91e8c', 'Mujer')]:
    mask = df['patient_gender'] == gen
    axes[3].scatter(df.loc[mask, 'umap_x'], df.loc[mask, 'umap_y'],
                    c=color, s=1.5, alpha=0.35, label=f'{nombre} ({mask.sum():,})')
axes[3].set_title('Genero', fontsize=13, fontweight='bold')
axes[3].legend(markerscale=6, fontsize=10)

for ax in axes:
    ax.set_xlabel('UMAP 1', fontsize=10)
    ax.set_ylabel('UMAP 2', fontsize=10)
    ax.grid(True, alpha=0.15)
    ax.tick_params(labelsize=8)

plt.suptitle('Proyecto Vena — Visualizacion UMAP del espacio de embeddings',
             fontsize=16, fontweight='bold', y=1.01)
plt.savefig(str(RUTA_RESULTADOS / 'umap_panel_completo.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Panel UMAP guardado.')

## Celda 15 — Analisis de clusters K-Means

Para cada cluster se reportan las metricas clinicas mas relevantes:
proporcion de sanos/enfermos, genero, edad y fuente del dataset.

In [ ]:
print(f'ANALISIS DE CLUSTERS K-MEANS (K={K_OPTIMO})')
print('=' * 70)

resumen_km = []
for k in range(K_OPTIMO):
    sub = df[df['cluster_kmeans'] == k]
    r = {
        'Cluster'    : k,
        'N'          : len(sub),
        'Pct_total'  : len(sub) / len(df) * 100,
        'Pct_enf'    : (sub['label'] == 1).mean() * 100,
        'Pct_sano'   : (sub['label'] == 0).mean() * 100,
        'Pct_M'      : (sub['patient_gender'] == 'M').mean() * 100,
        'Pct_F'      : (sub['patient_gender'] == 'F').mean() * 100,
        'Edad_media' : sub['patient_age'].mean(),
        'Edad_std'   : sub['patient_age'].std(),
        'Pct_NIH'    : (sub['source'] == 'NIH').mean() * 100,
        'Pct_CHEX'   : (sub['source'] == 'CHEX').mean() * 100,
    }
    resumen_km.append(r)

    perfil = 'ENFERMOS' if r['Pct_enf'] > 60 else ('SANOS' if r['Pct_sano'] > 60 else 'MIXTO')
    genero = 'MASCULINO' if r['Pct_M'] > 60 else ('FEMENINO' if r['Pct_F'] > 60 else 'MIXTO')
    edad   = 'JOVENES' if r['Edad_media'] < 40 else ('ADULTOS MAYORES' if r['Edad_media'] > 60 else 'ADULTOS')

    print(f'\nCluster {k} — {r["N"]:,} pacientes ({r["Pct_total"]:.1f}% del total)')
    print(f'  Estado  : {r["Pct_enf"]:.1f}% enfermos  |  {r["Pct_sano"]:.1f}% sanos   → {perfil}')
    print(f'  Genero  : {r["Pct_M"]:.1f}% hombres  |  {r["Pct_F"]:.1f}% mujeres → {genero}')
    print(f'  Edad    : {r["Edad_media"]:.1f} ± {r["Edad_std"]:.1f} años              → {edad}')
    print(f'  Fuente  : {r["Pct_NIH"]:.1f}% NIH  |  {r["Pct_CHEX"]:.1f}% CheXpert')

df_resumen_km = pd.DataFrame(resumen_km)
df_resumen_km.to_csv(str(RUTA_RESULTADOS / 'kmeans_resumen.csv'), index=False)
print('\nResumen guardado en kmeans_resumen.csv')

## Celda 16 — Radiografias reales por cluster K-Means


Esta celda muestra N_MUESTRAS_RX radiografias reales de cada cluster.
Permite ver visualmente que tipo de patron agrupo el modelo en cada grupo.
Las imagenes se muestran con su etiqueta real (sano/enfermo), edad y genero.

In [ ]:
def mostrar_rx_por_cluster(df, col_cluster, titulo_general, nombre_archivo):
    clusters = sorted([c for c in df[col_cluster].unique() if c != -1])
    n_cols   = N_MUESTRAS_RX
    n_rows   = len(clusters)

    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(n_cols * 2.8, n_rows * 3.2))
    if n_rows == 1:
        axes = axes[np.newaxis, :]

    paleta = sns.color_palette('tab10', len(clusters))

    for row, k in enumerate(clusters):
        sub      = df[df[col_cluster] == k]
        muestras = sub.sample(min(N_MUESTRAS_RX, len(sub)), random_state=SEMILLA)

        pct_enf  = (sub['label'] == 1).mean() * 100
        edad_med = sub['patient_age'].mean()
        pct_f    = (sub['patient_gender'] == 'F').mean() * 100
        color_borde = paleta[row]

        for col, (_, fila) in enumerate(muestras.iterrows()):
            ax = axes[row, col]
            try:
                img = Image.open(fila['ruta_completa']).convert('L')
                ax.imshow(img, cmap='gray', aspect='auto')
            except Exception:
                ax.text(0.5, 0.5, 'Error', ha='center', va='center',
                        transform=ax.transAxes)
                ax.set_facecolor('#111')

            estado = 'ENFERMO' if fila['label'] == 1 else 'SANO'
            color  = '#ff4444' if fila['label'] == 1 else '#44ff88'
            ax.set_title(
                f"{estado}  |  {int(fila['patient_age'])}a  {fila['patient_gender']}",
                fontsize=8, color=color, fontweight='bold', pad=3
            )
            for spine in ax.spines.values():
                spine.set_edgecolor(color_borde)
                spine.set_linewidth(2.5)
            ax.set_xticks([])
            ax.set_yticks([])

        # Etiqueta del cluster en el lado izquierdo
        axes[row, 0].set_ylabel(
            f'Cluster {k}\n{len(sub):,} px\n{pct_enf:.0f}% enf\n{edad_med:.0f}a med\n{pct_f:.0f}% F',
            fontsize=8, rotation=0, labelpad=60, va='center',
            color=[c for c in color_borde]
        )

    fig.suptitle(titulo_general, fontsize=15, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(str(RUTA_RESULTADOS / nombre_archivo),
                dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Imagen guardada: {nombre_archivo}')


mostrar_rx_por_cluster(
    df,
    col_cluster     = 'cluster_kmeans',
    titulo_general  = f'Proyecto Vena — Radiografias por Cluster K-Means (K={K_OPTIMO})',
    nombre_archivo  = 'kmeans_rx_por_cluster.png'
)

## Celda 17 — Heatmap de caracteristicas por cluster K-Means

Visualiza las metricas de cada cluster en un heatmap normalizado.
Permite identificar de un vistazo que diferencia a cada grupo.

In [ ]:
cols_heatmap = ['Pct_enf', 'Pct_F', 'Edad_media', 'Pct_NIH']
labels_hm    = ['% Enfermos', '% Mujeres', 'Edad media', '% NIH']

matriz = df_resumen_km[cols_heatmap].values.astype(float)
# Normalizar cada columna a [0,1] para comparacion visual
matriz_norm = (matriz - matriz.min(axis=0)) / (matriz.max(axis=0) - matriz.min(axis=0) + 1e-8)

fig, ax = plt.subplots(figsize=(10, max(4, K_OPTIMO * 0.8)))
im = ax.imshow(matriz_norm, cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=1)

ax.set_xticks(range(len(labels_hm)))
ax.set_xticklabels(labels_hm, fontsize=12)
ax.set_yticks(range(K_OPTIMO))
ax.set_yticklabels([f'Cluster {k}' for k in range(K_OPTIMO)], fontsize=11)

# Valores reales dentro de cada celda
for i in range(K_OPTIMO):
    for j, col in enumerate(cols_heatmap):
        val = df_resumen_km[col].iloc[i]
        fmt = f'{val:.1f}' if col != 'Edad_media' else f'{val:.0f}a'
        ax.text(j, i, fmt, ha='center', va='center',
                fontsize=10, fontweight='bold',
                color='white' if matriz_norm[i, j] > 0.6 else 'black')

plt.colorbar(im, ax=ax, label='Valor normalizado', shrink=0.8)
ax.set_title('Perfil de cada Cluster K-Means', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(str(RUTA_RESULTADOS / 'kmeans_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

## Celda 18 — Radiografias reales por cluster DBSCAN

In [ ]:
# Solo mostrar clusters reales (excluir ruido -1)
df_sin_ruido = df[df['cluster_dbscan'] != -1].copy()

if len(df_sin_ruido) > 0:
    mostrar_rx_por_cluster(
        df_sin_ruido,
        col_cluster    = 'cluster_dbscan',
        titulo_general = 'Proyecto Vena — Radiografias por Cluster DBSCAN',
        nombre_archivo = 'dbscan_rx_por_cluster.png'
    )
else:
    print('DBSCAN no encontro clusters validos con los parametros actuales.')
    print('Ajustar EPS en Celda 12 y volver a ejecutar.')

## Celda 19 — Comparacion final: K-Means vs DBSCAN

Tabla resumen que compara las metricas principales de ambos algoritmos.
Incluye metricas de calidad del clustering.

In [ ]:
# Metricas de calidad (sobre muestra para agilizar)
idx_eval = np.random.choice(len(X_pca), size=min(8000, len(X_pca)), replace=False)

labels_km_eval = df['cluster_kmeans'].values[idx_eval]
sil_km = silhouette_score(X_pca[idx_eval], labels_km_eval, random_state=SEMILLA)
db_km  = davies_bouldin_score(X_pca[idx_eval], labels_km_eval)

labels_db_eval = df['cluster_dbscan'].values[idx_eval]
mask_no_ruido  = labels_db_eval != -1

if mask_no_ruido.sum() > 100:
    sil_db = silhouette_score(X_pca[idx_eval][mask_no_ruido],
                              labels_db_eval[mask_no_ruido], random_state=SEMILLA)
    db_db  = davies_bouldin_score(X_pca[idx_eval][mask_no_ruido],
                                  labels_db_eval[mask_no_ruido])
else:
    sil_db = db_db = float('nan')

n_db_clusters = len(set(df['cluster_dbscan'].unique()) - {-1})
pct_ruido     = (df['cluster_dbscan'] == -1).mean() * 100

print('=' * 60)
print('COMPARACION: K-Means vs DBSCAN')
print('=' * 60)
print(f'{"Metrica":<30} {"K-Means":>12} {"DBSCAN":>12}')
print('-' * 60)
print(f'{"Clusters encontrados":<30} {K_OPTIMO:>12} {n_db_clusters:>12}')
print(f'{"Puntos ruido (%)":<30} {"0.0%":>12} {pct_ruido:>11.1f}%')
print(f'{"Silhouette Score":<30} {sil_km:>12.4f} {sil_db:>12.4f}')
print(f'{"Davies-Bouldin (menor=mejor)":<30} {db_km:>12.4f} {db_db:>12.4f}')
print('=' * 60)

ganador_sil = 'K-Means' if sil_km > sil_db else 'DBSCAN'
ganador_db  = 'K-Means' if db_km  < db_db  else 'DBSCAN'
print(f'\nMejor Silhouette   : {ganador_sil}')
print(f'Mejor Davies-Bouldin: {ganador_db}')

# Guardar comparacion
comparacion = pd.DataFrame({
    'Metrica'  : ['N_clusters', 'Pct_ruido', 'Silhouette', 'Davies_Bouldin'],
    'KMeans'   : [K_OPTIMO, 0.0, sil_km, db_km],
    'DBSCAN'   : [n_db_clusters, pct_ruido, sil_db, db_db]
})
comparacion.to_csv(str(RUTA_RESULTADOS / 'comparacion_kmeans_dbscan.csv'), index=False)
print('\nComparacion guardada en comparacion_kmeans_dbscan.csv')

## Celda 20 — Grafica comparativa visual

Panel final que resume visualmente la comparacion entre ambos algoritmos.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

metricas = ['Silhouette\n(mayor = mejor)', 'Davies-Bouldin\n(menor = mejor)']
vals_km  = [sil_km, db_km]
vals_db  = [sil_db, db_db]

x = np.arange(len(metricas))
w = 0.35

bars1 = axes[0].bar(x - w/2, vals_km, w, label='K-Means', color='steelblue', alpha=0.85)
bars2 = axes[0].bar(x + w/2, vals_db, w, label='DBSCAN',  color='coral',     alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(metricas, fontsize=11)
axes[0].set_title('Metricas de calidad del clustering', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3, axis='y')
for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2, h + 0.002,
                 f'{h:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Distribucion de tamanos de clusters
sizes_km = [( df['cluster_kmeans'] == k).sum() for k in range(K_OPTIMO)]
sizes_db = [(df[df['cluster_dbscan'] != -1]['cluster_dbscan'] == k).sum()
             for k in sorted(set(df['cluster_dbscan'].unique()) - {-1})]

axes[1].bar([f'K{k}' for k in range(K_OPTIMO)], sizes_km,
            color='steelblue', alpha=0.85, label='K-Means')
if sizes_db:
    ax2 = axes[1].twinx()
    ax2.bar([f'D{k}' for k in range(len(sizes_db))], sizes_db,
            color='coral', alpha=0.6, label='DBSCAN')
    ax2.set_ylabel('Tamano clusters DBSCAN', fontsize=10, color='coral')

axes[1].set_title('Tamano de clusters por algoritmo', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Tamano clusters K-Means', fontsize=10, color='steelblue')
axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle('Proyecto Vena — Comparacion K-Means vs DBSCAN',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(str(RUTA_RESULTADOS / 'comparacion_final.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Grafica comparativa guardada.')

## Celda 21 — Resumen final y archivos generados

Lista todos los archivos producidos por este notebook.

In [ ]:
print('=' * 60)
print('NOTEBOOK 03 — AGRUPACION COMPLETADO')
print('=' * 60)
print(f'\nDataset utilizado      : {len(df):,} radiografias (master_train.csv)')
print(f'Embeddings             : {embeddings.shape[1]} dimensiones por imagen')
print(f'K-Means K optimo       : {K_OPTIMO}')
print(f'DBSCAN clusters        : {n_db_clusters}')
print(f'DBSCAN ruido           : {pct_ruido:.1f}% de los puntos')
print(f'\nArchivos generados en  : {RUTA_RESULTADOS}')
print()
for archivo in sorted(RUTA_RESULTADOS.iterdir()):
    tam = archivo.stat().st_size / 1024
    unidad = 'KB' if tam < 1024 else 'MB'
    tam = tam if tam < 1024 else tam / 1024
    print(f'  {archivo.name:<45} {tam:>6.1f} {unidad}')

print()
print('Para subir al bucket:')
print('gsutil -m cp -r /home/user/checkpoints/agrupacion/ gs://vena-dataset/resultados_agrupacion/')